In [1]:
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.table import Table


## Data loads

### Folder parsing

In [2]:
data_folder = '/Users/philvanlane/Documents/lc_ae/data/'

### Data Load

In [3]:
readme = data_folder + 'Kim_thickdisk_ReadMe.txt'
data = data_folder + 'Kim_thickdisk_table6.dat'
data_formatted = data_folder + 'Kim_thickdisk.csv'

In [8]:
meta_headers = pd.read_csv(readme,
                               skiprows=64,
                               nrows=1,
                               header=None,
                              sep=r"\s+"
                         ).values[0]

In [9]:
meta_headers

array(['Bytes', 'Format', 'Units', 'Label', 'Explanations'], dtype=object)

In [20]:
meta_data = pd.read_fwf(readme,
                               widths=[9,6,7,13,100],
                               header=58,
                                nrows=16,
                               names=meta_headers
                         )

In [21]:
meta_data = meta_data.dropna(subset=['Bytes'])

In [22]:
meta_data

,Bytes,Format,Units,Label,Explanations
0,1- 19,I19,---,GaiaEDR3,Gaia EDR3 unique source identifier
1,21- 31,F11.7,deg,RAdeg,Right ascension (ICRS) at Ep=2016
2,33- 43,F11.7,deg,DEdeg,Declination (ICRS) at Ep=2016
3,45- 52,F8.4,mas,plx,Parallax
4,54- 59,F6.4,mas,e_plx,Standard error of parallax
5,61- 69,F9.3,mas/yr,pmRA,"Proper motion in right ascension direction,"
7,71- 79,F9.3,mas/yr,pmDE,Proper motion in declination direction
8,81- 86,F6.3,mag,Gmag,G-band mean magnitude
9,88- 92,F5.3,mag,G-RP,G-RP colour index
10,94- 98,F5.3,mag,E(G-RP),G-RP colour excess (1)


In [23]:
data_widths = []

for k in range(len(meta_data['Bytes'].values)):
    c = meta_data['Bytes'].values[k]
    try:
        i = c.index('-')
        start = int(c[0:i])
        end = int(c[i+1:])
        data_widths.append(end - start + 2)
    except:
        data_widths.append(2)

In [24]:
df = pd.read_fwf(data,
                               widths=data_widths,
                               header=None,
                               names=meta_data['Label'].values
                         )

In [25]:
df

,GaiaEDR3,RAdeg,DEdeg,plx,e_plx,pmRA,pmDE,Gmag,G-RP,E(G-RP),AG,[Fe/H]KNN,[Fe/H]grid
0,4489782790598521344,268.100928,11.066672,5.2962,0.0926,-70.637,-226.936,17.316,1.146,0.000,0.000,-0.218,-0.137
1,4490275509244724608,262.784331,8.347827,5.8721,0.0609,-157.284,-116.449,16.624,1.124,0.000,0.000,-0.091,-0.158
2,4489082161172907136,265.863353,9.275887,4.0087,0.0949,-11.188,-109.420,17.443,1.184,0.091,0.371,-0.149,-0.039
3,4490338005313126912,262.418249,8.530161,2.4762,0.1529,-47.539,-49.179,18.316,1.140,0.067,0.274,-0.188,-0.326
4,4489611640444691968,267.018765,10.210906,3.4534,0.1416,-78.226,-68.369,17.959,1.164,0.073,0.299,-0.178,-0.320
...,...,...,...,...,...,...,...,...,...,...,...,...,...
551209,4936389188137316480,30.614246,-52.366801,1.3055,0.1626,-0.314,-96.533,18.770,0.843,0.000,0.000,-2.242,-2.971
551210,4956444314588060800,30.756574,-43.340177,1.2610,0.1500,31.481,-35.613,18.754,0.843,0.000,0.000,-2.525,-2.879
551211,4957939650401850880,31.356174,-41.144315,1.3903,0.1509,36.018,-29.462,18.675,0.840,0.000,0.000,-2.144,-3.000
551212,4750552447230227840,45.487313,-48.003949,1.3100,0.1439,90.958,-43.238,18.640,0.843,0.000,0.000,-2.369,-2.851


In [26]:
df.to_csv(data_formatted, index=False)

In [40]:
ra_hours = data['RAh'].values  # RA hours
ra_minutes = data['RAm'].values  # RA minutes
ra_seconds = data['RAs'].values  # RA seconds

dec_degrees = data['DEd'].values  # Dec degrees
dec_minutes = data['DEm'].values  # Dec minutes
dec_seconds = data['DEs'].values  # Dec seconds

# Create SkyCoord object
coord = SkyCoord(ra=ra_hours*u.hour + ra_minutes*u.minute + ra_seconds*u.second,
                 dec=dec_degrees*u.deg + dec_minutes*u.arcmin + dec_seconds*u.arcsec, 
                 frame='icrs')

# Convert RA and Dec to degrees
ra_in_degrees = coord.ra.deg
dec_in_degrees = coord.dec.deg

data['RA_deg'] = ra_in_degrees
data['DE_deg'] = dec_in_degrees

In [38]:
data.to_csv(datafile_formatted)